In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os, glob, warnings
import numpy as np
import pandas as pd
import xarray as xr
from scipy.signal import butter, sosfilt
import matplotlib.pyplot as plt
from functools import reduce

warnings.filterwarnings("ignore")

# ── 0) your actual root (fix the “link_am334” typo!)
BASE_DIR   = "/home/am334/link_am334/praki/cmip6_data"

MODELS     = [
    
    "ACCESS-ESM1-5","ACCESS-CM2","CESM2","GFDL-ESM4","FGOALS-g3",
    "MRI-ESM2-0","MIROC6","CanESM5","GISS-E2-1-G","NorESM2-LM","NorESM2-MM",
    "HadGEM3-GC31-LL","UKESM1-1-LL","CMCC-ESM2","HadGEM3-GC31-MM",
    "MPI-ESM1-2-HR","INM-CM4-8","CanESM5-CanOE","GFDL-CM4", "CAS-ESM2-0", "IPSL-CM6A-LR"
]


SCENARIO   = "historical"
LAT_TARGET = 26.5

# low-pass
CUTOFF     = 1/120   # 10-yr cutoff (cycles/month)
LPF_ORDER  = 5

# which models use msftyz instead of msftmz
YZ_MODELS = {
    "CIESM","GFDL-CM4","GFDL-ESM4","HadGEM3-GC31-LL","CMCC-ESM2",
    "CMCC-CM2-HR4","CNRM-CM6-1-HR","EC-Earth3","HadGEM3-GC31-MM",
    "IPSL-CM6A-LR","IPSL-CM6A-MR1","UKESM1-1-LL"
}
# some models index the Atlantic basin differently
BASIN_IDX1_MODELS = {
    "MPI-ESM1-2-HR","CNRM-CM6-1-HR","EC-Earth3",
    "IPSL-CM6A-MR1","IPSL-CM6A-LR","E3SM-1-0"
}

# ── helper: lowpass ───────────────────────────────────────────────
def lowpass_filter(data, cutoff=CUTOFF, order=LPF_ORDER, fs=1.0):
    pad = int(2/cutoff)
    sos = butter(order, cutoff, "low", output="sos", fs=fs)
    padded = np.pad(data, ((pad,pad),(0,0),(0,0)), mode="reflect")
    return sosfilt(sos, padded, axis=0)[pad:-pad,:,:]

# ── helper: fix coords & pick Atlantic ───────────────────────────
def prepare_da(ds, var, model):
    da = ds[var]
    # 3-basin output?
    if "3basin" in da.dims:
        da = da.rename({"3basin":"basin"}).isel(basin=1)
    if "basin" in da.dims:
        idx = 1 if model in BASIN_IDX1_MODELS else 0
        da = da.isel(basin=idx)
    # unify vertical
    if "depth" in da.dims and "lev" not in da.dims:
        da = da.rename(depth="lev")
    if "olevel" in da.dims and "lev" not in da.dims:
        da = da.rename(olevel="lev")


    lev_units = da.lev.attrs.get("units", "").lower()
    if lev_units in ("cm", "centimeter", "centimeters"):
        da = da.assign_coords(lev=da.lev / 100)

    
    if model == "IPSL-CM6A-LR":
            da = da[:,:,:,0]
    
    if model == "IPSL-CM6A-LR":
        # IPSL gives nav_lat[y] as a 1-D array
        # attach it to the remaining 'y' dim, then rename
        nav = ds.nav_lat
        if "x" in nav.dims:
            nav = nav.isel(x=0)
        nav[-1] = 90
        da = da.rename({"y": "rlat"})
        # this works because dim 'rlat' now exists
        da.coords["rlat"] = ("rlat", nav.values)

    else:
        # for all other models, just rename whatever lat dim they use
        for old in ("y", "nav_lat", "lat", "latitude"):
            if old in da.dims:
                da = da.rename({old: "rlat"})
                break

                        
    # restrict & interp to regular grid
    da = da.sel(lev=slice(0,2500))
    da = da.interp(
        lev=np.arange(0,2501,100),
        rlat=np.arange(-30,70.1,2.5),
        method="linear"
    ).fillna(0)
    return da

# ── 1) read each model/member only from msftmz/msftyz dirs ───────
records = []

for model in MODELS:
    var = "msftyz" if model in YZ_MODELS else "msftmz"
    model_hist = os.path.join(BASE_DIR, model, SCENARIO)
    member_dirs = glob.glob(os.path.join(model_hist, "r*i*p*f*"))
    if not member_dirs:
        print(f"[skip] {model}: no member dirs under {model_hist}")
        continue

    for mdir in sorted(member_dirs):
        member = os.path.basename(mdir)
        var_dir = os.path.join(mdir, var)
        nc_files = sorted(glob.glob(os.path.join(var_dir, "*.nc")))
        if not nc_files:
            # no that variable for this member
            continue

        da_list = []
        for fn in nc_files:
            try:
                with xr.open_dataset(fn, use_cftime=True) as ds:
                    if var in ds:
                        da_list.append(prepare_da(ds, var, model))
            except Exception as e:
                print(f"[err] {model}/{member} → {e}")

        if not da_list:
            continue

        # concat in time, filter, pick max over depth at 35°N
        moc = xr.concat(da_list, dim="time")
        moc.values = lowpass_filter(moc.values)
        amoc = moc.sel(rlat=LAT_TARGET, method="nearest").max(dim="lev")

        df = amoc.to_dataframe(name="AMOC_26N").reset_index()
        df["model"]  = model
        df["member"] = member
        records.append(df)

    print(f"[✓] {model} → {len(member_dirs)} members scanned")

if not records:
    raise RuntimeError("No AMOC series generated; check your paths/vars.")

# ── 2) stack into (year, run) DataArray ──────────────────────────
def cf_to_years(times):
    return np.array([t.year for t in times], dtype=int)

da_list = []
runs    = []

for df in records:
    run_id = f"{df.model.iloc[0]}_{df.member.iloc[0]}"
    runs.append(run_id)

    years = cf_to_years(df["time"].values)
    da = (xr.DataArray(df["AMOC_26N"].values,
                       coords={"time": years},
                       dims=("time",))
            .groupby("time").mean()
            .rename("year")
            .rename(time="year")
            .expand_dims(run=[run_id]))
    da_list.append(da)

common_years = sorted(reduce(set.intersection,
                            [set(d.year.values) for d in da_list]))
da_stack = xr.concat([d.sel(year=common_years) for d in da_list],
                     dim="run")
da_stack = da_stack.assign_coords(run=runs)

# ── 3) compute linear trends per run ────────────────────────────
periods = {
    "1870–1985": slice(1870, 1985),
    "1870–2014": slice(1870, 2014),
    "1985–2014": slice(1985, 2014),
    "1900–2014": slice(1900, 2014),
    "1900–1985": slice(1900, 1985),
}

def calc_trends(da_period):
    yrs = da_period.year.values
    slopes = {}
    for r in da_period.run.values:
        y = da_period.sel(run=r).values
        m = np.isfinite(y)
        if m.sum() < 2:
            slopes[r] = np.nan
        else:
            slopes[r], _ = np.polyfit(yrs[m], y[m], 1)
    return slopes

all_trends = {name: calc_trends(da_stack.sel(year=periods[name]))
              for name in periods}

# ── 4) histograms (Sv/decade) ────────────────────────────────────
fig, axarr = plt.subplots(1, 3, figsize=(15,4), sharey=True)
for ax, (name, trdict) in zip(axarr, all_trends.items()):
    vals = np.array(list(trdict.values())) * 10 / 10**9
    ax.hist(vals, bins=20, edgecolor="black")
    ax.set_title(name)
    ax.set_xlabel("Sv per decade")
axarr[0].set_ylabel("Members")
fig.suptitle("AMOC (26.5°N) Trends by Ensemble Member")
plt.tight_layout(rect=[0,0,1,0.95])
plt.show()


In [ ]:
import xarray as xr
import numpy as np

ds = xr.open_dataset("/home/am334/link_am334/moc_mmodel/bnn_real_world_rec/amoc_from_ssh.nc")

da = ds["amoc_from_ssh_mu"]

# select 2004–2023 inclusive, drop years to it be same as RAPID
da_2004_2023 = da.sel(time=slice("2006-01-01", "2023-12-31"))

# drop NaNs along time if any
da_2004_2023 = da_2004_2023.dropna("time")

time = da_2004_2023["time"]

# fractional years: e.g. 2004.5, 2005.25, ...
t_years = (time.dt.year + (time.dt.dayofyear - 1) / 365.25).astype(float)


y = da_2004_2023.values  # AMOC values

# linear fit y = a * t_years + b
a, b = np.polyfit(t_years, y, 1)

trend_per_year = a
total_change = a * (float(t_years[-1]) - float(t_years[0]))

print("Linear trend:", trend_per_year*100, "per century")
print("Total change over period:", total_change)
print("Start year:", float(t_years[0]), "End year:", float(t_years[-1]))



In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import xarray as xr
from scipy.signal import butter, sosfilt
import matplotlib.pyplot as plt
from functools import reduce


def lowpass_filter(data, cutoff_freq, order=5, fs=1, pad=2):
    sos = butter(order, cutoff_freq, btype='low', output='sos', fs=fs)
    if data.ndim == 1:
        padded = np.pad(data, (pad, pad), mode='reflect')
        filtered = sosfilt(sos, padded)[pad:-pad]
    elif data.ndim == 2:
        padded = np.pad(data, ((pad, pad), (0, 0)), mode='reflect')
        filtered = sosfilt(sos, padded, axis=0)[pad:-pad, :]
    elif data.ndim == 3:
        padded = np.pad(data, ((pad, pad), (0, 0), (0, 0)), mode='reflect')
        filtered = sosfilt(sos, padded, axis=0)[pad:-pad, :, :]
    else:
        raise ValueError(f"Unsupported data dimensions: {data.ndim}")
    return np.nan_to_num(filtered)

    

ds_moc = xr.open_dataset('/home/am334/link_am334/moc_mmodel/moc_vertical.nc')

ds_moc_t = xr.open_dataset('/home/am334/link_am334/moc_mmodel/moc_transports (3).nc')
rapid_t = ds_moc_t['moc_mar_hc10'].resample(time='1M').mean()
rapid_t_wo_lp = lowpass_filter(rapid_t, 1/24, 5, 1, 48)


rapid = ds_moc['stream_function_mar'].resample(time='1M').mean()
rapid_max = rapid.max(dim='depth')
rapid_wo_lp = lowpass_filter(rapid_max, 1/24, 5, 1, 48)



# Строим линейный тренд на допустимом интервале (без LPF)
t_r = np.arange(len(rapid_wo_lp[24:]), dtype=float)
y_r = rapid_wo_lp[24:]
# Уберём явные NaN
valid = np.isfinite(y_r)
t_r = t_r[valid]; y_r = y_r[valid]
if y_r.size >= 3:
    slope_r, intercept_r = np.polyfit(t_r, y_r, 1)
    trend_r = slope_r * t_r + intercept_r


    

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.colors import Normalize, ListedColormap
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FormatStrFormatter
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# =============================
# Inputs
# =============================
PERIOD1 = "1900–1985"
CSV_PATH = "/home/am334/link_am334/praki/cmip6_data/_hist_trends_hist+ssp585_members_only/trends_1992-2023_Sv_per_decade.csv"

SCALE_PERIOD1 = 1e-7
SCALE_DEC2CEN = 10.0

OBS1_MEAN = 0.396
OBS1_CI   = (OBS1_MEAN - 0.683, OBS1_MEAN + 0.683)
CEASER2018 = -2.375

OBS2_MEAN_DEC = -3.559
OBS2_CI_DEC   = (OBS2_MEAN_DEC - 1.240, OBS2_MEAN_DEC + 1.240)
OBS2_MEAN = OBS2_MEAN_DEC
OBS2_CI   = (OBS2_CI_DEC[0], OBS2_CI_DEC[1])

# SST→AMOC line + spread for the BOTTOM panel
SST2_MEAN = -3.074
SST2_ERR  = 1.218
SST2_CI   = (SST2_MEAN - SST2_ERR, SST2_MEAN + SST2_ERR)

# =============================
# ASR map (TOP panel ordering + coloring)
# =============================
asr_order_map = {
    "ACCESS-CM2": 0.011857092227438408, "ACCESS-ESM1-5": 0.014893554203697335,
    "CAS-ESM2-0": 0.020470713524869528, "CESM2": 0.021538743949348533,
    "CMCC-ESM2": 0.004133331907451014, "CanESM5": 0.012011685045444682,
    "CanESM5-CanOE": 0.011990572698048374, "FGOALS-g3": -0.003239418791663882,
    "GFDL-CM4": 0.01800142727816291, "GFDL-ESM4": 0.010082614519302242,
    "GISS-E2-1-G": 0.01572108248807887, "HadGEM3-GC31-LL": 0.012554644085396499,
    "HadGEM3-GC31-MM": 0.012094279665669491, "INM-CM4-8": 0.005641965077352932,
    "IPSL-CM6A-LR": -0.0005154370575548654, "MIROC6": 0.004068202260233284,
    "MPI-ESM1-2-HR": 0.0011229192646631282, "MRI-ESM2-0": 0.020548003819369283,
    "NorESM2-LM": 0.022282111815345628, "NorESM2-MM": 0.012232974654473214,
    "UKESM1-1-LL": 0.022485546181942553,
}

# =============================
# Mean AMOC per model (BOTTOM panel ordering + coloring)
# =============================
AMOC_MEAN_MAP = {
    "ACCESS-CM2": 17.7,
    "ACCESS-ESM1-5": 18.2,
    "CanESM5": 11.7,
    "CanESM5-CanOE": 11.5,
    "CAS-ESM2-0": 18.1,
    "CESM2": 18.0,
    "CMCC-ESM2": 18.1,
    "FGOALS-g3": 30.3,
    "GFDL-CM4": 19.7,
    "GFDL-ESM4": 18.4,
    "GISS-E2-1-G": 24.2,
    "HadGEM3-GC31-LL": 15.5,
    "HadGEM3-GC31-MM": 16.4,
    "INM-CM4-8": 19.9,
    "IPSL-CM6A-LR": 12.1,
    "MIROC6": 15.1,
    "MRI-ESM2-0": 18.1,
    "MPI-ESM1-2-HR": 16.9,
    "NorESM2-LM": 20.3,
    "NorESM2-MM": 20.9,
    "UKESM1-1-LL": 13.7,
}

# =============================
# Helper functions
# =============================
def normalize_model_name(m: str) -> str:
    if m == "ACCESS-ESM1--5":
        return "ACCESS-ESM1-5"
    return m

def trends_dict_to_df(all_trends: dict) -> pd.DataFrame:
    rows = []
    for period, runs in all_trends.items():
        for run_id, val in runs.items():
            model = run_id.split("_", 1)[0] if "_" in run_id else run_id
            model = normalize_model_name(model)
            rows.append({"period": period, "model": model, "member": run_id, "trend": val})
    df = pd.DataFrame(rows)
    return df.replace([np.inf, -np.inf], np.nan).dropna(subset=["trend", "model"])

def build_order_by_asr(models: pd.Series, asr_map: dict) -> list[str]:
    present = [normalize_model_name(m) for m in models.dropna().unique().tolist()]
    scored = [(m, asr_map.get(m, np.inf)) for m in present]
    scored.sort(key=lambda t: t[1])
    return [m for m, _ in scored]

def build_order_by_amoc(models: pd.Series, amoc_map: dict, ascending: bool = True) -> list[str]:
    present = [normalize_model_name(m) for m in models.dropna().unique().tolist()]
    scored = [(m, amoc_map.get(m, np.nan)) for m in present]
    scored.sort(key=lambda t: (np.isnan(t[1]), t[1] if ascending else -t[1]))
    return [m for m, _ in scored]

def add_all_violin(df: pd.DataFrame, label="All models") -> pd.DataFrame:
    pooled = df.copy()
    pooled["model"] = label
    return pd.concat([df, pooled], ignore_index=True)

def make_soft_cmap(base_name="viridis", whiten=0.22, N=256):
    """
    Create a softer version of a cmap by blending toward white.
    Use for BOTH violins and colorbar => perfect match.
    """
    base = mpl.colormaps[base_name]
    cols = base(np.linspace(0, 1, N))
    cols[:, :3] = (1 - whiten) * cols[:, :3] + whiten * 1.0
    return ListedColormap(cols, name=f"soft_{base_name}")

def add_colorbar_center_right(
    ax,
    sm,
    label: str,
    *,
    ticks=None,
    fmt=None,
    width="2.2%",
    height="44%",
    pad=0.03,
    label_fs=14,
    tick_fs=12,
):
    cax = inset_axes(
        ax,
        width=width,
        height=height,
        loc="center left",
        bbox_to_anchor=(1 + pad, 0.1, 1, 1),
        bbox_transform=ax.transAxes,
        borderpad=0.0,
    )
    cb = plt.colorbar(sm, cax=cax)
    cb.set_label(label, fontsize=label_fs, labelpad=6)
    cb.ax.tick_params(labelsize=tick_fs, length=2.5, width=0.8)

    if ticks is not None:
        cb.set_ticks(ticks)
    if fmt is not None:
        cb.ax.yaxis.set_major_formatter(FormatStrFormatter(fmt))

    cb.outline.set_linewidth(0.8)
    cb.outline.set_edgecolor("0.6")
    for spine in cb.ax.spines.values():
        spine.set_linewidth(0.8)
        spine.set_edgecolor("0.6")
    return cb

def build_palette_and_mappable_seq(
    model_order,
    index_map,
    *,
    cmap,
    robust=False,
    p_lo=5.0,
    p_hi=95.0,
    missing_color="0.80",
    all_models_color="#D9DEE7",
):
    """
    Sequential mapping.
    If robust=True, use percentile clipping to spread colors when values are clustered.
    Returns: palette(dict), sm(ScalarMappable), vmin, vmax
    """
    if isinstance(cmap, str):
        cmap = mpl.colormaps[cmap]

    vals = np.array([index_map.get(normalize_model_name(m), np.nan) for m in model_order], float)
    finite = np.isfinite(vals)
    if not finite.any():
        palette = {m: missing_color for m in model_order}
        palette["All models"] = all_models_color
        return palette, None, None, None

    v = vals[finite]

    if robust and v.size >= 3:
        vmin = float(np.nanpercentile(v, p_lo))
        vmax = float(np.nanpercentile(v, p_hi))
        if not (vmax > vmin):
            vmin = float(np.nanmin(v))
            vmax = float(np.nanmax(v))
    else:
        vmin = float(np.nanmin(v))
        vmax = float(np.nanmax(v))

    norm = Normalize(vmin=vmin, vmax=vmax, clip=True)

    palette = {}
    for m in model_order:
        vv = index_map.get(normalize_model_name(m), np.nan)
        palette[m] = cmap(norm(vv)) if np.isfinite(vv) else missing_color
    palette["All models"] = all_models_color

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    return palette, sm, vmin, vmax

def panel_plot(
    ax,
    data,
    model_order,
    *,
    obs_mean,
    obs_ci,
    ylabel="Sv / century",
    ceaser=None,
    obs_color=None,
    # index-based violin coloring
    violin_index_map=None,
    violin_index_label="",
    violin_cmap=None,
    robust_colors=False,
    robust_p_lo=5.0,
    robust_p_hi=95.0,
    add_violin_colorbar=False,
    colorbar_fmt=None,
    # fonts
    ylab_fs=18,
    # single-point fix
    single_point_fix=True,
):
    order_plus_all = model_order + ["All models"]
    d = data.copy()
    d["model"] = d["model"].map(normalize_model_name)
    d["model"] = pd.Categorical(d["model"], categories=order_plus_all, ordered=True)

    palette = None
    sm = None
    vmin = vmax = None

    if violin_index_map is not None:
        if violin_cmap is None:
            violin_cmap = mpl.colormaps["viridis"]
        palette, sm, vmin, vmax = build_palette_and_mappable_seq(
            model_order=model_order,
            index_map=violin_index_map,
            cmap=violin_cmap,
            robust=robust_colors,
            p_lo=robust_p_lo,
            p_hi=robust_p_hi,
        )
        sns.violinplot(
            data=d, x="model", y="trend", order=order_plus_all,
            inner="box", cut=0, scale="width", linewidth=0.8,
            palette=palette, saturation=1.0, ax=ax
        )
    else:
        sns.violinplot(
            data=d, x="model", y="trend", order=order_plus_all,
            inner="box", cut=0, scale="width", linewidth=0.8,
            color="#D9DEE7", ax=ax
        )

    sns.stripplot(
        data=d, x="model", y="trend", order=order_plus_all,
        size=2.2, jitter=0.22, alpha=0.85, linewidth=0,
        color="#5A6A85", ax=ax
    )

    # Model means
    mmean = d[d["model"] != "All models"].groupby("model")["trend"].mean()
    xticks = {m: i for i, m in enumerate(order_plus_all)}
    ax.scatter(
        [xticks[m] for m in mmean.index],
        mmean.values,
        marker="D", s=20, color="black", zorder=3, label="_nolegend_"
    )

    # Obs band + mean
    if (obs_mean is not None) and (obs_ci is not None) and (obs_color is not None):
        low, high = obs_ci
        ax.axhspan(low, high, color=obs_color, alpha=0.08, zorder=0)
        ax.axhline(obs_mean, color=obs_color, linewidth=2.0, zorder=2, label="_nolegend_")

    # SST index (dotted)
    if ceaser is not None:
        ax.axhline(ceaser, color="0.25", linestyle=(0, (1, 2)), linewidth=1.8, zorder=2, label="_nolegend_")

    # Single-point fix: show color even when violin collapses to a line
    if single_point_fix:
        counts = d.groupby("model")["trend"].size()

        def _model_color(m):
            if palette is not None and m in palette:
                return palette[m]
            return "#D9DEE7"

        for m in model_order:
            if m in counts.index and int(counts[m]) == 1:
                y = float(d.loc[d["model"] == m, "trend"].iloc[0])
                x = xticks[m]
                c = _model_color(m)
                ax.hlines(y, x - 0.33, x + 0.33, color=c, linewidth=9.0, zorder=2)
                ax.scatter(x, y, s=80, facecolor=c, edgecolor="0.25", linewidth=0.8, zorder=4)

    # Styling
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, fontsize=ylab_fs)
    ax.set_xticklabels(order_plus_all, rotation=30, ha="right")
    ax.grid(axis="y", linewidth=0.7, alpha=0.99)
    sns.despine(ax=ax)

    # Colorbar ticks: evenly spaced (no vmin/0/vmax bullshit)
    if add_violin_colorbar and (sm is not None) and violin_index_label:
        ticks = None
        if (vmin is not None) and (vmax is not None) and np.isfinite(vmin) and np.isfinite(vmax) and (vmax > vmin):
            ticks = np.linspace(vmin, vmax, 5)
            ticks[np.isclose(ticks, 0.0, atol=1e-14)] = 0.0  # avoid "-0.0000"
        add_colorbar_center_right(
            ax, sm, violin_index_label,
            ticks=ticks, fmt=colorbar_fmt,
            label_fs=14, tick_fs=12,
            width="2.2%", height="44%", pad=0.03
        )

# =============================
# IMPORTANT: must exist upstream
# =============================
assert "all_trends" in globals(), "Define all_trends upstream before running this script."
assert "slope_r" in globals(), "Define slope_r upstream before running this script."

# =============================
# Build datasets
# =============================
df_all = trends_dict_to_df(all_trends)

df1 = df_all[df_all["period"] == PERIOD1][["model", "member", "trend"]].copy()
df1["trend"] = df1["trend"] * SCALE_PERIOD1

df2_raw = pd.read_csv(CSV_PATH)
df2 = (
    df2_raw.rename(columns={"run": "member", "slope_Sv_per_decade": "trend"})
          .assign(model=lambda d: d["member"].str.split("_", n=1).str[0].map(normalize_model_name))
)
df2 = df2.replace([np.inf, -np.inf], np.nan).dropna(subset=["trend", "model"])
df2["trend"] = df2["trend"] * SCALE_DEC2CEN

order_top = build_order_by_asr(df1["model"], asr_order_map)
order_bot = build_order_by_amoc(df2["model"], AMOC_MEAN_MAP, ascending=True)

df1_plus = add_all_violin(df1, label="All models")
df2_plus = add_all_violin(df2, label="All models")

# =============================
# Plot setup
# =============================
mpl.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 16,
    "legend.fontsize": 14,
    "xtick.labelsize": 10,
    "ytick.labelsize": 12,
    "axes.linewidth": 0.9,
    "figure.dpi": 150,
})
sns.set(style="ticks", context="paper")

max_models = max(len(order_top), len(order_bot))
fig_w = 0.62 * max(18, max_models + 2)
fig_h = 10.2
fig, axes = plt.subplots(2, 1, figsize=(fig_w, fig_h), sharey=False)

SOFT_VIRIDIS = make_soft_cmap("viridis", whiten=0.22)
SOFT_CIVIDIS = make_soft_cmap("cividis", whiten=0.18)

# =============================
# TOP panel: ASR_HD coloring
# =============================
panel_plot(
    axes[0], df1_plus, order_top,
    obs_mean=OBS1_MEAN, obs_ci=OBS1_CI,
    ylabel="AMOC at 26N trend \n 1900--1985 (Sv / century)",
    ceaser=CEASER2018,
    obs_color="blue",
    violin_index_map=asr_order_map,
    violin_index_label=r"$ASR_{HD}$ (W m$^{-2}$ month$^{-1}$)",
    violin_cmap=SOFT_VIRIDIS,
    robust_colors=False,
    add_violin_colorbar=True,
    colorbar_fmt="%.3f",
    ylab_fs=18,
)

# =============================
# BOTTOM panel: color by mean AMOC strength
# =============================
panel_plot(
    axes[1], df2_plus, order_bot,
    obs_mean=OBS2_MEAN, obs_ci=OBS2_CI,
    ylabel="AMOC at 26N trend \n 1993--2023 (Sv / century)",
    obs_color="orange",
    violin_index_map=AMOC_MEAN_MAP,
    violin_index_label="Mean AMOC (Sv)",
    violin_cmap=SOFT_CIVIDIS,
    robust_colors=True,
    robust_p_lo=5,
    robust_p_hi=95,
    add_violin_colorbar=True,
    colorbar_fmt="%.0f",
    ylab_fs=18,
)

# Add SST→AMOC band+line to bottom (no legend here)
axes[1].axhspan(SST2_CI[0], SST2_CI[1], color="blue", alpha=0.08, zorder=0)
axes[1].axhline(SST2_MEAN, color="blue", linewidth=1.8, zorder=3, label="_nolegend_")

# Your extra dotted line (kept)
axes[1].axhline(
    -2.22,
    color="0.25", linestyle=(0, (1, 2)),
    linewidth=1.6,
    zorder=2
)

# Limits
axes[0].set_ylim(-6, 6)
axes[1].set_ylim(-30, 5)

# Panel labels: a, b
for ax, lab in zip(axes, ["a", "b"]):
    ax.text(
        0.01, 0.98, lab,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=18, fontweight="bold"
    )

# =============================
# Legend (fixed: consistent handles/labels)
# =============================
nn_sst_handle = Line2D([0], [0], color="blue", linewidth=2.5, label="NN SST")
nn_ssh_handle = Line2D([0], [0], color="orange", linewidth=2.5, label="NN SSH")
sst_index_handle = Line2D([0], [0], color="0.25", linestyle=(0, (1, 2)), linewidth=1.8, label="SST index")

fig.legend(
    [nn_sst_handle, nn_ssh_handle, sst_index_handle],
    ["DL SST", "DL SSH", "SST index"],
    loc="lower center", bbox_to_anchor=(0.5, 0.93),
    ncol=3, frameon=False, fontsize=14
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.14)

# Save before show (so it always writes the file)
fig.savefig("fig_amoc_violin_top_asr_bottom_amocmean.png", bbox_inches="tight")
plt.show()
